In [1]:
import napari
from numpy import spacing
from skimage.io import imread
import glob
from wip_util import zero_pad_index, load_iters, get_np_points, reverse_emitters, smap_csv_to_emitters
import numpy as np
import decode

start_iter = 645
end_iter = 646

frames_per_iter = 250

start_frame = start_iter * frames_per_iter
end_frame = end_iter * frames_per_iter

c:\Users\bnort\miniconda3\envs\decode_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load frames, for this dataset sets of 250 frames are saved in tif sequences of 250 frames.  Each sequence is a single iteration.

iters_path = r'E:\Cryo-PALM Data for Brian\James processing ASCII and RAW Data files\PALM_Run2_slab_0001_488nm_Frames_(Processed_Slab)'
iter_pattern = lambda id:rf'{iters_path}\3dpalm488nm_iter_{zero_pad_index(id, 4)}_0001_ch0_cam1_stack0000_405nm_0000000msec_*msecabs_000x_000y_000z_0001t.tif'
frames = load_iters(start_iter, end_iter, iter_pattern)
print(frames.shape)

loading iter for id: 645
(250, 800, 800)


In [ ]:
smap_input_file = r'D:\Janelia_slm_data\Janelia PALM RUN2\SMAP_Result_iter_645.csv'
smap_emitters = smap_csv_to_emitters(smap_input_file, 100, True)
print(smap_emitters)

EmitterSet
::num emitters: 37477
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 249
::spanned volume: [    9.036018    11.95644  -1970.      ] - [ 794.8877  795.826  2000.    ]
EmitterSet
::num emitters: 37477
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 249
::spanned volume: [    9.036018    11.95644  -1970.      ] - [ 794.8877  795.826  2000.    ]


In [ ]:
hesslab_input_file = r'C:\Users\bnort\work\Janelia_slm\data\James processing ASCII and RAW Data files\2a - PALM_Run2_Slab_0001_488nm_constrained_Gaussian_fitting_gauss_half_width_3_with_winding_file_RAW_UNPROCESSED_PALM_LOCALIZATIONS_ASCII.hdf5'
hesslab_emitters=decode.EmitterSet.load(hesslab_input_file)
print(hesslab_emitters)
print()
# filter based on start and end frame
hesslab_emitters = hesslab_emitters[(hesslab_emitters.frame_ix >= start_frame) & (hesslab_emitters.frame_ix < end_frame)]
print(hesslab_emitters)
hesslab_emitters.xyz_px[:,0].max(), hesslab_emitters.xyz_px[:,1].max(), hesslab_emitters.xyz_px[:,2].max()


EmitterSet
::num emitters: 41570847
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 0 - 289749
::spanned volume: [ 2.32811e-01  1.93751e-01 -8.00000e+02] - [800.    799.475 600.   ]

EmitterSet
::num emitters: 24993
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 161250 - 161499
::spanned volume: [  23.4343     2.68076 -799.951  ] - [799.325 772.354 599.663]


(tensor(799.3250, dtype=torch.float64),
 tensor(772.3540, dtype=torch.float64),
 tensor(599.6630, dtype=torch.float64))

In [ ]:
# note decode emitters are stored in a file with a start and end iter number, which may not be the same as the start and iter numbers used to load the frames.
# so we explicitly define it
# Naming schemes are a work in progress, so this may change in the future.

chunk_string = f"{zero_pad_index(start_iter, 4)}-{zero_pad_index(650, 4)}"
decode_input_file = rf'D:\Janelia_slm_data\Data_2025_05_27\network_CMOS_C13440_20CUd_05-01_chunk_{chunk_string}.csv'
print(start_frame, end_frame)
decode_emitters=decode.EmitterSet.load(decode_input_file)
print(decode_emitters)

161250 161500


c:\users\bnort\work\janelia_slm\code\decode\decode\generic\emitter.py:280: UserWarning: For .csv files, implicit usage of .load() is discouraged. Please use 'decode.utils.emitter_io.load_csv' explicitly.
  warnings.warn("For .csv files, implicit usage of .load() is discouraged. "


EmitterSet
::num emitters: 783247
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 161250 - 162749
::spanned volume: [-4.1167417e-01 -5.7897305e-01 -1.1264720e+03] - [ 798.91437  798.9928  1039.0311 ]


In [ ]:
sigma_x_high_threshold = 50
sigma_y_high_threshold = 50
sigma_z_high_threshold = 100
prob_threshold = 0.5
decode_emitters = decode_emitters[
    (decode_emitters.xyz_sig_nm[:, 0] <= sigma_x_high_threshold)
    * (decode_emitters.xyz_sig_nm[:, 1] <= sigma_x_high_threshold)
    * (decode_emitters.xyz_sig_nm[:, 2] <= sigma_z_high_threshold)
    * (decode_emitters.prob >= prob_threshold)
    ]



In [ ]:
print(hesslab_emitters)
sigma_x_high_threshold = 2000 
sigma_y_high_threshold = 2000
sigma_z_high_threshold = 10000
prob_threshold = 0.0
coord_limit=((500,700),(100,200))
coord_limit=((0,500),(0,700))

print("0,1", coord_limit[0][0], coord_limit[0][1])

hesslab_emitters_filtered = hesslab_emitters[
    (hesslab_emitters.xyz_sig_nm[:, 0] <= sigma_x_high_threshold)
    * (hesslab_emitters.xyz_sig_nm[:, 1] <= sigma_x_high_threshold)
    * (hesslab_emitters.xyz_sig_nm[:, 2] <= sigma_z_high_threshold)
    * (hesslab_emitters.prob >= prob_threshold) 
    #* (hesslab_emitters.xyz_px[:, 0] <= 100) #coord_limit[0][0])
    ]

print(hesslab_emitters_filtered)



EmitterSet
::num emitters: 24993
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 161250 - 161499
::spanned volume: [  23.4343     2.68076 -799.951  ] - [799.325 772.354 599.663]
0,1 0 500
EmitterSet
::num emitters: 24993
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 161250 - 161499
::spanned volume: [  23.4343     2.68076 -799.951  ] - [799.325 772.354 599.663]


In [ ]:
viewer = napari.Viewer()
viewer.add_image(frames[0])

<Image layer 'Image' at 0x250d936c9d0>

In [ ]:
reverse_emitters(decode_emitters, frames.shape[:2], 1, 1)

In [ ]:
decode_points = get_np_points(decode_emitters, start_frame, start_frame+100, True)
print(decode_points.shape)
decode_points = decode_points[decode_points[:,0] == decode_points[0,0]]
print(decode_points.shape)
decode_points_t = decode_points[:,1:4]
decode_points_t = decode_points_t[:, [2, 0, 1]]  # Reorder to (z, y, x)

(10159, 4)
(97, 4)


In [ ]:
smap_points = get_np_points(smap_emitters, 0, 250, True)
print(smap_points.shape)
smap_points = smap_points[:, 1:4]

(37346, 4)


In [ ]:
print(hesslab_emitters)
hesslab_points = get_np_points(hesslab_emitters, start_frame, start_frame+100, True)
print(hesslab_points.shape)
hesslab_points = hesslab_points[hesslab_points[:,0] == hesslab_points[0,0]]
print(hesslab_points.shape)
hesslab_points_t = hesslab_points[:,1:4]
hesslab_points_t = hesslab_points_t[:, [2, 1, 0]]  # Reorder to (z, y, x)

EmitterSet
::num emitters: 24993
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 161250 - 161499
::spanned volume: [  23.4343     2.68076 -799.951  ] - [799.325 772.354 599.663]
(9901, 4)
(109, 4)


In [ ]:
decode_points_t[:,0] = decode_points_t[:,0] - decode_points_t[:,0].min()
hesslab_points_t[:,0] = hesslab_points_t[:,0] - hesslab_points_t[:,0].min()

In [ ]:
viewer.add_points(decode_points_t, size=10, face_color='blue', name='decode emitters (t)')
viewer.add_points(hesslab_points_t, size=10, face_color='red', name='hesslab emitters (t)')

<Points layer 'hesslab emitters (t)' at 0x2510b9c15e0>

In [ ]:
print(hesslab_emitters)
hesslab_points = get_np_points(hesslab_emitters, start_frame, start_frame+100, True)
hesslab_points.shape
hesslab_points[:,0].max(), hesslab_points[:,1].max(), hesslab_points[:,2].max(), hesslab_points[:,3].max()

EmitterSet
::num emitters: 24993
::xy unit: px
::px size: tensor([130., 130.])
::frame range: 161250 - 161499
::spanned volume: [  23.4343     2.68076 -799.951  ] - [799.325 772.354 599.663]


(161349.0, 798.42, 772.354, 598.252)

In [ ]:
decode_points_t[:,0].min(), decode_points_t[:,1].min(), decode_points_t[:,2].min()

(-819.17822265625, 31.8754940032959, 41.91632080078125)

In [ ]:

hesslab_points_t[:,0].max(), hesslab_points_t[:,1].max(), hesslab_points_t[:,2].max()

(508.158, 793.857, 692.392)